# Preprocessing — Video → Dataset Frames

## 1 · Imports & paths

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

# Make ../src importable
ROOT = Path.cwd().resolve()
if (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.extract_frames import (
    extract_frames_from_dir, summarize,
    letterbox, laplacian_variance,
    DEFAULT_SAMPLE_RATE, DEFAULT_TARGET_SIZE, DEFAULT_BLUR_THRESHOLD,
)

RAW_DIR     = ROOT / 'data' / 'raw'
FRAMES_DIR  = ROOT / 'data' / 'frames'
OUTPUTS_DIR = ROOT / 'outputs'

plt.rcParams['figure.figsize'] = (12, 4)
print('OpenCV:', cv2.__version__, '  NumPy:', np.__version__)
print('RAW   :', RAW_DIR)
print('FRAMES:', FRAMES_DIR)


## 2 · Discover videos

In [ ]:
VIDEO_EXTS = ('.mp4', '.avi', '.mov', '.mkv', '.webm')

def list_videos(root: Path) -> list[tuple[str, Path]]:
    """Return [(label, path), ...] for every video under root."""
    found: list[tuple[str, Path]] = []
    for p in sorted(root.rglob('*')):
        if p.is_file() and p.suffix.lower() in VIDEO_EXTS:
            label = f'{p.parent.name}/{p.stem}'
            found.append((label, p))
    return found

VIDEOS = list_videos(RAW_DIR)
for label, path in VIDEOS:
    print(f'  {label:40s}  {path}')
print(f'\nTotal: {len(VIDEOS)} videos')


## 3 · Video metadata

In [ ]:
def video_info(path: Path) -> dict:
    """Open a video, read metadata, close cleanly."""
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise IOError(f'Cannot open video: {path}')
    info = {
        'path':   str(path),
        'width':  int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
        'height': int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
        'fps':    float(cap.get(cv2.CAP_PROP_FPS) or 0),
        'frames': int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0),
    }
    cap.release()
    return info

INFOS = [(label, video_info(p)) for label, p in VIDEOS]
for label, info in INFOS:
    print(f'  {label:40s}  {info["width"]:>4}x{info["height"]:<4}  '
          f'{info["fps"]:5.1f} fps  {info["frames"]:>5} frames')


## 4 · Sample frames per video

In [ ]:
def grab_frames(path: Path, indices) -> list[np.ndarray]:
    """Return RGB frames at given indices."""
    cap = cv2.VideoCapture(str(path))
    out: list[np.ndarray] = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, frame = cap.read()
        if ok:
            out.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
    return out

def show_grid(frames, title, cols=4):
    rows = (len(frames) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3 * rows))
    axes = np.atleast_1d(axes).ravel()
    for ax, f, i in zip(axes, frames, range(len(frames))):
        ax.imshow(f); ax.set_title(f'frame #{i}'); ax.axis('off')
    for a in axes[len(frames):]: a.axis('off')
    fig.suptitle(title); plt.tight_layout(); plt.show()

# Show 5 evenly-spaced frames per video, biased toward late timestamps
# (fire/smoke events typically happen later in these clips)
for label, path in VIDEOS[:6]:  # cap at 6 videos to keep notebook short
    info = video_info(path)
    if info['frames'] < 2:
        continue
    fracs = [0.10, 0.50, 0.80, 0.95, 0.99]
    idxs = [min(info['frames'] - 1, int(info['frames'] * f)) for f in fracs]
    show_grid(grab_frames(path, idxs), label)


## 5 · Colour-space EDA

In [ ]:
label, path = VIDEOS[0]
info = video_info(path)
idx = max(0, int(info['frames'] * 0.9))
rgb = grab_frames(path, [idx])[0]
bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(rgb);                axes[0].set_title('RGB');  axes[0].axis('off')
axes[1].imshow(hsv);                axes[1].set_title('HSV');  axes[1].axis('off')
axes[2].imshow(lab);                axes[2].set_title('LAB');  axes[2].axis('off')
fig.suptitle(f'Colour spaces — {label}'); plt.tight_layout(); plt.show()


## 6 · Sharpness check

In [ ]:
label, path = VIDEOS[0]
cap = cv2.VideoCapture(str(path))
variances: list[float] = []
while True:
    ok, frame = cap.read()
    if not ok: break
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    variances.append(laplacian_variance(gray))
cap.release()

variances = np.array(variances)
fig, ax = plt.subplots(1, 1, figsize=(10, 3))
ax.plot(variances, lw=0.7)
ax.axhline(DEFAULT_BLUR_THRESHOLD, color='r', ls='--', label=f'threshold={DEFAULT_BLUR_THRESHOLD}')
ax.set_title(f'Laplacian variance per frame — {label}')
ax.set_xlabel('frame'); ax.set_ylabel('variance'); ax.legend()
plt.tight_layout(); plt.show()

print(f'Mean: {variances.mean():.1f}   Median: {np.median(variances):.1f}   '
      f'<threshold: {(variances < DEFAULT_BLUR_THRESHOLD).mean():.1%}')


## 7 · Letterbox preview

In [ ]:
lb = letterbox(cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR), target=DEFAULT_TARGET_SIZE)
lb_rgb = cv2.cvtColor(lb, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(rgb);    axes[0].set_title(f'Original  {rgb.shape[1]}x{rgb.shape[0]}'); axes[0].axis('off')
axes[1].imshow(lb_rgb); axes[1].set_title(f'Letterboxed  {lb.shape[1]}x{lb.shape[0]}'); axes[1].axis('off')
plt.tight_layout(); plt.show()


## 8 · Extract all frames

In [ ]:
SAMPLE_RATE = 10  # ← bump this down (e.g. 5) for more frames, up for fewer

results = extract_frames_from_dir(
    input_dir=RAW_DIR,
    output_dir=FRAMES_DIR,
    sample_rate=SAMPLE_RATE,
    target_size=640,
    blur_threshold=100.0,
    hash_distance=4,
    workers=1,  # bump for parallelism (uses ProcessPoolExecutor)
)
summary = summarize(results)
print('\n=== Extraction summary ===')
for k, v in summary.items():
    print(f'  {k}: {v}')


## 9 · Final per-class frame counts

In [ ]:
print('\nFrames extracted per class:')
total = 0
for sub in sorted(FRAMES_DIR.iterdir()):
    if not sub.is_dir(): continue
    n = sum(1 for _ in sub.rglob('*.jpg'))
    print(f'  {sub.name:8s}: {n:5d}')
    total += n
print(f'  {"TOTAL":8s}: {total:5d}')
